## 1_imputation_examples.ipynb
### Fairness and Compliance in AI
### University of Oklahoma

This script demonstrates the use of MICE to impute missing values.  It downloads the UCI Adult Census dataset, cleans it and describes it.  It then performs Little's MCAR Test to determine fairness attributes that are MAR.  It then uses MICE to impute those values.  Finally, it saves the resulting dataframe to a CSV file for use in other processes.

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
import random

random.seed(42)


# ---------------------------------------------------------
# 1. Read in the UCI adult dataset and load it into a pandas DataFrame
# ---------------------------------------------------------
try:
    cols = ['age', 'work', 'fnlwgt', 'edu', 'num_edu', 'mar', 'occ', 'rel', 'race', 'sex', 'gain', 'loss', 'hpw', 'country', 'target']
    df = pd.read_csv('adult.data', names=cols, sep=r',\s*', engine='python')
except Exception as e:
    print(f"Error occurred while trying to read local file: {e}")
    raise

# Replace the '?' with NaN values
df.replace('?', np.nan, inplace=True)

# Look at the first few rows of the dataset
df.head()

In [ ]:
# ---------------------------------------------------------
# 2. Cleanup the dataset and check for missing values
# ---------------------------------------------------------

# The mar variable differentiates between married categories. We simplify it to just married.
df['mar'] = df['mar'].replace({'Married-civ-spouse': 'Married',
                               'Married-spouse-absent': 'Married',
                               'Married-AF-spouse': 'Married'})

# Collapse Amer-Indian-Eskimo and Other into a single category "Other" for simplicity
df['race'] = df['race'].replace({'Amer-Indian-Eskimo': 'Other', 'Other': 'Other'})

# Collapse country into US vs non-US for simplicity, preserving missing values
# If country is missing (NaN), keep NaN instead of assigning 'Non-US'.
df['country'] = df['country'].apply(
    lambda x: 'US' if x == 'United-States' else (np.nan if pd.isna(x) else 'Non-US')
)

# Eliminate unnecessary categories:
#     fnlwgt is a weighting column used for extrapolating counts to the general population
#     work and occ are somewhat redundant -- keep work
#     edu is not as informative as num_edu but is directly correlated
df.drop(columns=['fnlwgt', 'work', 'edu'], inplace=True)

def summarize_dataframe(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100
    })
    return summary.reset_index().rename(columns={"index": "column"})

summary = summarize_dataframe(df)
print("\n=== COLUMN SUMMARY ===")
print(summary)

We see that occ (occupation) and country have missingness.  We check to
see if it is MCAR using Little's MCAR test.

In [ ]:
# ---------------------------------------------------------
# 3. Perform Little's MCAR test to check if missingness is MCAR, MAR, or MNAR
# ---------------------------------------------------------

def run_mcar_test(df, missing_col):
    """Run Little's MCAR test by logistic regression of missingness indicator against all other columns.

    Parameters:
    - df: pandas DataFrame
    - missing_col: name of column for missingness indicator

    Returns:
    - fitted statsmodels Logit result object, or None on failure
    """
    df = df.copy()
    df['is_missing'] = df[missing_col].isnull().astype(int)

    predictor_cols = [c for c in df.columns if c != missing_col and c != 'is_missing']

    df_temp = df.dropna(subset=predictor_cols)
    if df_temp.empty:
        raise ValueError('No rows left after dropping NA in predictors; cannot run MCAR test.')

    X_mcar = sm.add_constant(pd.get_dummies(df_temp[predictor_cols], drop_first=True))
    X_mcar = X_mcar.astype(float)

    y_mcar = df_temp['is_missing']

    try:
        model = sm.Logit(y_mcar, X_mcar).fit(method='bfgs', maxiter=1000, disp=False)
        return model
    except Exception as e:
        print(f"Model fitting failed: {e}")
        return None


# Example usage: test whether 'occ' is MCAR.  Remove 'target' from the MCAR analysis since it's the label.
missing_cols = ['occ', 'country']  # List of columns with missing values to test
for col in missing_cols:
    mcar_model = run_mcar_test(df.drop(columns=["target"]), col)
    if mcar_model is not None:
        print(f"Little's MCAR Test Results for {col} (all variables included):")
        print(mcar_model.summary())
    else:
        print(f"Little's MCAR test for {col} could not be completed.")

The missingness for occ and country is MAR with respect to fairness variables, so we will impute using MICE

In [ ]:
# ---------------------------------------------------------
# 4. Impute the missing values with MICE
#    We create two separate functions: one for imputing categorical variables 
#    and one for imputing numeric variables.
#    Setting deterministic=True will use the mean imputation from MICE instead 
#    of sampling from the posterior distribution, which can be useful for testing and debugging.
# ---------------------------------------------------------

def mice_impute_numeric(df, num_vars, label=None, deterministic=False):
    """
    MICE-impute numeric variables using all other predictors.
    If label is provided, it is excluded from predictors.

    parameters:
    - df: pandas DataFrame
    - num_vars: list of numeric variable names to impute
    - label: name of label column to exclude from predictors (optional)
    - deterministic: whether to use deterministic imputation (optional)
    """

    # 1. Drop label if provided
    if label is not None and label in df.columns:
        df_features = df.drop(columns=[label])
    else:
        df_features = df.copy()

    # 2. Identify categorical + numeric predictors
    categorical_cols = df_features.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = df_features.select_dtypes(include=["int64", "float64"]).columns.tolist()

    # 3. One-hot encode categorical predictors ONLY
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoded_cats = ohe.fit_transform(df_features[categorical_cols])

    # 4. Combine numeric predictors + encoded categorical predictors
    numeric_matrix = df_features[numeric_cols].values
    full_matrix = np.hstack([numeric_matrix, encoded_cats])

    # 5. Run MICE
    mice = IterativeImputer(random_state=42, max_iter=5)
    if deterministic:
        mice = IterativeImputer(
            random_state=42,
            sample_posterior=False,
            max_iter=5
        )
    else:
        mice = IterativeImputer(random_state=42, max_iter=10)

    imputed_full = mice.fit_transform(full_matrix)

    # 6. Extract imputed numeric variables (first block of columns)
    imputed_numeric = imputed_full[:, :len(numeric_cols)]

    # 7. Insert imputed numeric vars back into original df
    df_result = df.copy()
    for i, var in enumerate(numeric_cols):
        if var in num_vars:
            df_result[var] = imputed_numeric[:, i]

    return df_result


In [ ]:
def mice_impute_categorical(df, cat_vars, label=None, deterministic=False):
    """
    Fast MICE imputation for categorical variables using Ordinal Encoding.
    - No one-hot explosion
    - Much faster than one-hot MICE

    parameters:
    - df: pandas DataFrame
    - cat_vars: list of categorical variable names to impute
    - label: name of label column to exclude from predictors (optional)
    - deterministic: whether to use deterministic imputation
    """

    # 1. Drop label if provided
    if label is not None and label in df.columns:
        df_features = df.drop(columns=[label])
    else:
        df_features = df.copy()

    # 2. Identify categorical + numeric predictors
    categorical_cols = df_features.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = df_features.select_dtypes(include=["int64", "float64"]).columns.tolist()

    # 3. Ordinal encode ALL categorical predictors
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    encoded_cats = enc.fit_transform(df_features[categorical_cols])

    # 4. Combine numeric + encoded categorical predictors
    full_matrix = np.hstack([df_features[numeric_cols].values, encoded_cats])

    # 5. Run MICE on the compact numeric matrix
    mice = IterativeImputer(random_state=42, max_iter=5)
    if deterministic:
        mice = IterativeImputer(
            random_state=42,
            sample_posterior=False,
            max_iter=5
        )
    else:
        mice = IterativeImputer(random_state=42, max_iter=10)

    imputed_full = mice.fit_transform(full_matrix)

    # 6. Extract imputed categorical block (last columns)
    imputed_cats = imputed_full[:, len(numeric_cols):]

    # 7. Decode back to category labels
    decoded_cats = enc.inverse_transform(imputed_cats)

    # 8. Insert imputed categorical vars back into original df
    df_result = df.copy()
    decoded_df = pd.DataFrame(decoded_cats, columns=categorical_cols)

    for var in cat_vars:
        df_result[var] = decoded_df[var]

    return df_result


In [ ]:
# Generate a new dataframe with imputed values for 'occ' and 'country'
# Because they are categorical, we use the mice_impute_categorical function,
# and we exclude 'target' from the predictors.
df_imputed = mice_impute_categorical(df, cat_vars=['occ', 'country'], label='target', deterministic=True)

In [ ]:
# Check the summary of the new dataframe to confirm imputation worked
print("\n=== COLUMN SUMMARY FOR NEW DATAFRAME ===")
print(summarize_dataframe(df_imputed))

Compare the change in value counts between the original dataframe and the new dataframe for occ and country

In [ ]:
def compare_occ_counts(df1, df2, var):
    """
    Compares change in distribution of imputed variable before and after 
    imputation by showing counts side-by-side.

    parameters:
    - df1: original dataframe with missing values
    - df2: new dataframe with imputed values
    - var: name of the variable to compare (default: "occ")
    """

    # Groupby counts for each dataframe
    g1 = df1.groupby(var).size().reset_index(name="count_df1")
    g2 = df2.groupby(var).size().reset_index(name="count_df2")

    # Merge side-by-side
    comparison = g1.merge(g2, on=var, how="outer").fillna(0)

    # Sort for readability
    return comparison.sort_values(var)


In [ ]:
comparison = compare_occ_counts(df, df_imputed, var="occ")
print("\n=== OCCUPATION COUNTS BEFORE AND AFTER IMPUTATION ===")
print(comparison)

In [ ]:
comparison = compare_occ_counts(df, df_imputed, var="country")
print("\n=== COUNTRY COUNTS BEFORE AND AFTER IMPUTATION ===")
print(comparison)

In [ ]:
# Save the dataframe with imputed values to a new CSV file
df_imputed.to_csv("adult_imputed.csv", index=False)